<a href="https://colab.research.google.com/github/Titantus/Truth-Zero-C/blob/main/T0C_Unified_Master_Showcase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **T0C Unified Master Showcase**
### *The Universal Rendering Engine: Geometry → Coherence → Utility*

---

## **Executive Summary**

**T0C (Theory of Zero-Torque Coherence)** models physical phenomena as torque packets routed through finite-resolution geometric nozzles on a κ-canvas. The heart of the system is the **η-Selector** — a multi-axis Gaussian resolver that maps geometric detuning (Δθ), cloud mismatch (Δχ), and frequency mismatch (Δf) into a single routing probability η.

This η determines the dominant mode:
- **Loop (c¹)** — rigidity, persistence, mass-like behavior  
- **Straight (c²)** — propagation, transparency, efficient transport  
- **Residue (c⁰)** — heat, dissipation, “fog”

By optimizing alignment with high-η paths, T0C predicts and resolves anomalies where standard models fail.

### **Live Demonstration**

The interactive 4-panel dashboard showcases T0C in action:

- **Ice VII/X Acoustic Anomaly** — η spike at ~64 GPa matching the observed elastic transition  
- **Material Coherence Profiles** — sharp η peaks near the tetrahedral lock (109.47°)  
- **Phase-Clash** — rigidity (Loop) vs transparency (Straight) competition  
- **Sodium Siphon** — ~85% heat reduction and >200% relative coherence gain via optimized routing  

**Current Status @ 64.0 GPa (Ice VII/X)**:  
Δθ = –0.1416° | η = 0.7591 | **Residue / Loop Regime** (approaching coherence lock)

---

## **Strategic Value**

T0C converts geometric “anomalies” into engineering levers:
- Predict strain-induced transparency thresholds  
- Resolve high-pressure phase transitions (e.g., Ice VII → X)  
- Design low-loss thermal and energy systems (Sodium Siphon)

**Next Steps (Phase I–III)**:
1. Overlay T0C predictions on real Brillouin and calorimetric datasets  
2. Release open-source T0C Python SDK  
3. Prototype siphon-based heat exchangers or electrodes  

**Document Version**: v7.5 · **Primary Vector**: Vector-S (Coherence Optimization)  
**System Status**: **Stable — Registry-Driven**

---

## **Core Model Assumptions**

- Torque packets routed through geometric nozzles on a finite-resolution canvas (κ)  
- η-Selector as the universal routing probability  
- Geometric detuning (Δθ) as the primary driver of coherence  
- CCZ saturation at p ≤ 0.0497 triggers resolution collapse and averaged behavior  

**Short Verdict**: T0C is an ambitious, internally consistent engineering-style framework that reframes physics as a rendering discipline. It excels as a specification for reproducible simulations and experiments; tighter numerical grounding and fully specified protocols will strengthen its scientific credibility.

In [ ]:
# @title Code Cell One:  Setup
from __future__ import annotations

import numpy as np
import json
import os
from pathlib import Path
from typing import Any, Dict, Tuple

import autograd.numpy as agnp
from autograd import grad
from autograd.numpy.numpy_boxes import ArrayBox # Import ArrayBox directly
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
import ipywidgets as widgets # Added ipywidgets import

# -------------------------------------------------------------------
# Plot styling (dark theme)
# -------------------------------------------------------------------
plt.style.use("dark_background")
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.edgecolor": "#444444",
    "grid.color": "#222222",
    "figure.facecolor": "#000000",
    "axes.facecolor": "#000000",
    "text.color": "#ffffff",
    "axes.labelcolor": "#ffffff",
    "xtick.color": "#ffffff",
    "ytick.color": "#ffffff",
})

# -------------------------------------------------------------------
# T0C Routing Engine (CCZ-aware)
# -------------------------------------------------------------------
class T0CEngine:
    """Core routing engine with precision vs. CCZ saturation modes."""

    def __init__(self, constants: Dict[str, Any]):
        self.c = constants
        cosmo = constants.get("cosmology_and_saturation", {})

        # CCZ parameters
        self.pc = cosmo.get("kappa_sat_pc", 0.0497)
        self.alpha_p = cosmo.get("ccz_default_alpha", 200.0)

        # Normalization scales
        p_scale = cosmo.get("p_scale_defaults", {})
        self.theta_norm = p_scale.get("ThetaNorm", 180.0)
        self.f_norm = p_scale.get("FNorm", 1e14)
        self.chi_norm = p_scale.get("ChiNorm", 1.0)

        # Gaussian sigmas (precision regime)
        self.sigma_theta = float(self.c.get("sigma_theta", 0.2))
        self.sigma_chi = float(self.c.get("sigma_chi", 0.05))
        self.sigma_f = float(self.c.get("sigma_f", 0.01))

    def calculate_proximity(self, d_theta: Any, d_chi: Any, d_f: Any) -> Any:
        """Radial proximity metric p."""
        p = agnp.sqrt(
            (d_theta / self.theta_norm)**2 +
            (d_chi / self.chi_norm)**2 +
            (d_f / self.f_norm)**2
        )
        return p

    def select_eta(self, d_theta: Any, d_chi: Any, d_f: Any) -> Tuple[Any, str, Any]:
        """Main routing selector: Gaussian or CCZ-averaged."""
        p = self.calculate_proximity(d_theta, d_chi, d_f)

        # Use the imported ArrayBox for type checking
        if isinstance(p, (np.ndarray, ArrayBox)) and agnp.any(p <= self.pc):
            # Saturated Regime (CCZ Drag / Resolution Collapse)
            # Use agnp.where to handle conditional operations with array-like inputs
            eta_ccz = agnp.exp(-self.alpha_p * (p ** 2))
            eta_prec = agnp.clip(agnp.exp(-(
                (d_theta ** 2) / (2 * self.sigma_theta ** 2) +
                (d_chi ** 2) / (2 * self.sigma_chi ** 2) +
                (d_f ** 2) / (2 * self.sigma_f ** 2)
            )), 1e-9, 1.0)

            # Combine based on condition using agnp.where
            eta = agnp.where(p <= self.pc, eta_ccz, eta_prec)
            mode = "CCZ_AVERAGED" if agnp.any(p <= self.pc) else "PRECISION_RESOLVED"
        else:
            # Precision Regime or single float case
            if p <= self.pc:
                eta = agnp.exp(-self.alpha_p * (p ** 2))
                mode = "CCZ_AVERAGED"
            else:
                exponent = -(
                    (d_theta ** 2) / (2 * self.sigma_theta ** 2) +
                    (d_chi ** 2) / (2 * self.sigma_chi ** 2) +
                    (d_f ** 2) / (2 * self.f_norm ** 2) # Corrected sigma_f here
                )
                eta = agnp.clip(agnp.exp(exponent), 1e-9, 1.0)
                mode = "PRECISION_RESOLVED"

        return eta, mode, p

    def eta_avg(self, p: Any) -> Any:
        """Explicit η_avg for Hubble drag, singularity, etc."""
        return agnp.exp(-self.alpha_p * (p ** 2))


# -------------------------------------------------------------------
# Registry Loading
# -------------------------------------------------------------------
REGISTRY_FILE = Path("T0C —  REGISTRY.json")   # Adjust if running locally or in Colab

def load_t0c_registry(path: os.PathLike | str) -> Tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any], Dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Registry not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    meta = data.get("meta", {})
    constants = meta.get("constants", {})
    elements = data.get("elements", {})
    molecules = data.get("molecules", {})

    return data, constants, elements, molecules


def compute_element_dynamics(el: Dict[str, Any]) -> Dict[str, Any]:
    """Add derived fields (eta_peak, flip_risk, etc.)."""
    gear = el.get("gear_assignment", "metal")
    is_tetra = gear == "tetra"
    el = dict(el)
    el["eta_peak"] = 0.95 if is_tetra else 0.01
    el["flip_risk"] = 0.01 if is_tetra else 1.0
    return el


# -------------------------------------------------------------------
# Global State & Initialization
# -------------------------------------------------------------------
T0C_REGISTRY: Dict[str, Any] | None = None
T0C_CONSTANTS: Dict[str, Any] | None = None
ELEMENTS: Dict[str, Any] | None = None
MOLECULES: Dict[str, Any] | None = None
t0c_engine: T0CEngine | None = None

try:
    T0C_REGISTRY, T0C_CONSTANTS, ELEMENTS, MOLECULES = load_t0c_registry(REGISTRY_FILE)

    # Enrich elements with derived dynamics
    ELEMENTS = {k: compute_element_dynamics(v) for k, v in ELEMENTS.items()}

    # Create the engine
    t0c_engine = T0CEngine(T0C_CONSTANTS)

    print(f"✅ T0C Engine Initialized — {len(ELEMENTS)} elements loaded from registry v{T0C_REGISTRY.get('meta', {}).get('registry_version', 'unknown')}")
    print(f"✅ T0CEngine ready with CCZ logic (kappa_sat_pc = {t0c_engine.pc:.4f}, alpha_p = {t0c_engine.alpha_p})")

except Exception as e:
    T0C_REGISTRY = None
    T0C_CONSTANTS = {}
    ELEMENTS = {}
    MOLECULES = {}
    t0c_engine = None
    print(f"❌ Registry load or engine initialization failed: {e}")


# -------------------------------------------------------------------
# Autograd-compatible selector (for gradients in later cells)
# -------------------------------------------------------------------
def eta_selector_autograd(d_theta: Any, d_chi: Any, d_f: Any) -> Any:
    """Public differentiable wrapper."""
    if t0c_engine is None:
        raise RuntimeError("T0CEngine not initialized")
    # The select_eta method now handles autograd types directly
    eta, _, _ = t0c_engine.select_eta(d_theta, d_chi, d_f)
    return agnp.array(eta)


# -------------------------------------------------------------------
# Geometrically Derived Engineering Parameters
# -------------------------------------------------------------------
# Define constants for magic numbers identified in the review
DEFAULT_D_CHI = 0.01
DEFAULT_D_F = 0.001
RIGIDITY_EXPONENT = 1.8
STABILITY_ZONE_MIN = 0.9
SIPHON_DISTANCE_NORMALIZATION = 90.0

if T0C_CONSTANTS:
    angle_ratio = T0C_CONSTANTS.get('theta_siphon', 70.53) / T0C_CONSTANTS.get('theta_tetra', 109.47)
    detuning_factor = agnp.exp( - (T0C_CONSTANTS.get('bounce_gap', 0.14122) / angle_ratio)**2 ) # Changed to agnp.exp

    NONLINEAR_COP_EXPONENT = (
        T0C_CONSTANTS.get('saturation_threshold', 0.95) *
        T0C_CONSTANTS.get('omega_c', 4.0) *
        (1 + T0C_CONSTANTS.get('back_pressure_coefficient', 0.002)) *
        (1 / (angle_ratio + 1e-6)) *
        (1 / (T0C_CONSTANTS.get('cosmology_and_saturation', {}).get('kappa_sat_pc', 0.0497) + 1e-6)) *
        T0C_CONSTANTS.get('N_spokes', 20) *
        detuning_factor
    )

    COP_BASE_MULTIPLIER = 2.065
    DEFAULT_HEAT_LOSS_FACTOR = 0.85

    print(f"✅ Geometrically derived NONLINEAR_COP_EXPONENT: {NONLINEAR_COP_EXPONENT:.4f}")
    print(f"   (angle_ratio={angle_ratio:.4f}, detuning_factor={detuning_factor:.6f})")


In [ ]:
# @title Code Cell 2: Unified T0C Interactive Lab (4-Panel Dashboard)

def eta_selector(d_theta: Any, d_chi: Any, d_f: Any, constants_ignored: Any) -> Any:
    """Helper to bridge old eta_selector calls to the T0CEngine."""
    if t0c_engine is None:
        raise RuntimeError("T0CEngine not initialized")
    eta, _, _ = t0c_engine.select_eta(d_theta, d_chi, d_f)
    return eta

def unified_t0c_dashboard(
    selected_materials=None,
    pressure_gpa: float = 64.0,
    p_scale_ice: float = 1.871,
    siphon_angle_offset: float = 0.0,
    std_freq_err: float = 0.05,
    opt_freq_err: float = 0.001,
):
    """
    Render the 4‑panel T0C interactive dashboard:
      1) Ice VII/X acoustic anomaly vs pressure
      2) Material coherence (η vs angle)
      3) Phase‑clash: rigidity vs transparency
      4) Sodium siphon COP comparison

    Parameters:
        selected_materials (list, optional): List of material symbols (e.g., ['C', 'Al']) to display coherence profiles. Defaults to ['C', 'Al'].
        pressure_gpa (float): Current pressure in GPa for the Ice VII/X anomaly plot. Defaults to 64.0.
        p_scale_ice (float): Scaling factor for pressure in the Ice VII/X anomaly calculation. Defaults to 1.871.
        siphon_angle_offset (float): Angular offset for the siphon angle, affecting COP calculation. Defaults to 0.0.
        std_freq_err (float): Standard frequency error for the 'Standard' siphon system. Defaults to 0.05.
        opt_freq_err (float): Optimized frequency error for the 'T0C-Optimized' siphon system. Defaults to 0.001.
    """
    if selected_materials is None:
        selected_materials = ['C', 'Al']

    # Handle case where ELEMENTS might be empty or not initialized
    if not ELEMENTS:
        print("Warning: ELEMENTS registry is empty or not initialized. Material plots will be skipped.")
        selected_materials = []

    # --- Figure scaffold ---
    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("T0C Unified Rendering Engine — Interactive Lab",
                 fontsize=18, color='white')
    colors = ['#00FFCC', '#FF3366', '#3399FF', '#FFFF00']

    # ---------- Panel 1: Ice VII/X Acoustic Anomaly ----------
    ax = axs[0, 0]
    pressure_range = np.linspace(0.0, 120.0, 300)

    delta_theta_0 = -4.97122
    align_p = 1.0 - 1.0 / (1.0 + pressure_range / p_scale_ice)
    delta_theta_p = delta_theta_0 * (1.0 - align_p)

    # Vectorized η evaluation
    # eta_ice = np.array([
    #     eta_selector(dt, 0.01, 0.001, T0C_CONSTANTS) for dt in delta_theta_p
    # ])
    eta_ice = t0c_engine.select_eta(delta_theta_p, DEFAULT_D_CHI, DEFAULT_D_F)[0]

    ax.plot(pressure_range, eta_ice, color='cyan', label='T0C η (H₂O)')
    ax.axvline(pressure_gpa, color='red', ls='--',
               label=f'Current P = {pressure_gpa:.1f} GPa')
    ax.axvline(64.0, color='white', ls=':', alpha=0.6,
               label='Predicted Transition (64 GPa)')
    ax.set_title("Ice VII/X Acoustic Anomaly")
    ax.set_xlabel("Pressure (GPa)")
    ax.set_ylabel("Routing Probability η")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Precompute index and live η for status block
    idx_p = int(np.argmin(np.abs(pressure_range - pressure_gpa)))
    detuning_at_p = float(delta_theta_p[idx_p])
    current_eta_ice = float(eta_ice[idx_p])

    # ---------- Panel 2: Material Coherence Profiles ----------
    ax = axs[0, 1]
    strain_range = np.linspace(30.0, 120.0, 200)

    for i, mat in enumerate(selected_materials):
        el = ELEMENTS.get(mat, {})
        theta_eq = el.get('theta_eq', 109.47)

        # Vectorized over strain_range
        # delta_theta = strain_range - theta_eq
        # eta_mat = np.array([
        #     eta_selector(dt, 0.01, 0.001, T0C_CONSTANTS) for dt in delta_theta
        # ])
        delta_theta_mat = strain_range - theta_eq
        eta_mat = t0c_engine.select_eta(delta_theta_mat, DEFAULT_D_CHI, DEFAULT_D_F)[0]

        ax.plot(
            strain_range,
            eta_mat,
            label=f"{mat} η",
            color=colors[i % len(colors)],
            lw=2,
        )

    ax.axvline(T0C_CONSTANTS['theta_tetra'], color='white', ls=':',
               label='Tetra Lock')
    ax.axvline(T0C_CONSTANTS['theta_siphon'], color='gold', ls=':',
               label='Siphon Pivot')
    ax.set_title("Material Coherence (η vs Angle)")
    ax.set_xlabel("Geometric Angle θ (°)")
    ax.set_ylabel("Routing Probability η")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ---------- Panel 3: Phase‑Clash (Rigidity vs Transparency) ----------
    ax = axs[1, 0]

    siphon_theta_const = T0C_CONSTANTS['theta_siphon']
    siphon_distance = np.abs(strain_range - siphon_theta_const) / SIPHON_DISTANCE_NORMALIZATION

    for i, mat in enumerate(selected_materials):
        el = ELEMENTS.get(mat, {})
        theta_eq = el.get('theta_eq', 109.47)

        delta_theta = strain_range - theta_eq
        eta_mat = t0c_engine.select_eta(delta_theta, DEFAULT_D_CHI, DEFAULT_D_F)[0]

        rigidity = eta_mat ** RIGIDITY_EXPONENT
        transparency = eta_mat * (1.0 - siphon_distance)

        color = colors[i % len(colors)]
        ax.plot(
            strain_range,
            rigidity,
            '--',
            color=color,
            alpha=0.7,
            label=f"{mat} Rigidity",
        )
        ax.plot(
            strain_range,
            transparency,
            color=color,
            label=f"{mat} Transparency",
        )

    ax.fill_between(
        strain_range,
        STABILITY_ZONE_MIN,
        1.0,
        color='green',
        alpha=0.15,
        label='Stability Zone',
    )
    ax.set_title("Phase‑Clash: Rigidity vs Transparency")
    ax.set_xlabel("Geometric Angle θ (°)")
    ax.set_ylabel("Mode Magnitude")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ---------- Panel 4: Sodium Siphon (Energy Efficiency) ----------
    ax = axs[1, 1]
    siphon_theta = T0C_CONSTANTS['theta_siphon'] + siphon_angle_offset

    systems = {
        'Standard': {
            'theta': 109.0,
            'f_err': std_freq_err,
            'color': '#444444',
        },
        'T0C-Optimized': {
            'theta': siphon_theta,
            'f_err': opt_freq_err,
            'color': '#00FF00',
        },
    }

    for label, params in systems.items():
        delta_theta_sys = params['theta'] - siphon_theta
        eta_sys = float(
            eta_selector(delta_theta_sys, DEFAULT_D_CHI, params['f_err'], T0C_CONSTANTS)
        )
        cop = eta_sys * COP_BASE_MULTIPLIER # Use constant COP_BASE_MULTIPLIER
        ax.bar(
            label,
            cop,
            color=params['color'],
            alpha=0.85,
            width=0.6,
        )
        ax.text(
            label,
            cop + 0.05,
            f"{cop:.2f}",
            ha='center',
            color='white',
        )

    ax.set_title("Sodium Siphon: Coefficient of Performance")
    ax.set_ylabel("COP (Relative Coherence Gain)")
    ax.grid(axis='y', alpha=0.3)

    # ---------- Layout + Render ----------
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # ---------- Live Status Summary ----------
    bounce_gap = T0C_CONSTANTS.get('bounce_gap', 0.141)
    status = (
        "COHERENCE LOCK (Transition)"
        if abs(detuning_at_p) <= bounce_gap
        else "RESIDUE / LOOP REGIME"
    )

    display(HTML(f"""
    <div style="background:#111; padding:12px; border:1px solid #444;
                border-radius:6px; color:#fff; font-family:monospace;">
        <b>Live T0C Status @ {pressure_gpa:.1f} GPa</b><br>
        Ice VII/X Detuning: {detuning_at_p:.4f}° |
        η = {current_eta_ice:.4f} |
        <span style="color:#0f0;">{status}</span>
    </div>
    """))


# --- Interactive Controls ---

# Default material selection, handle empty ELEMENTS
default_material_options = list(ELEMENTS.keys()) if ELEMENTS else ['No elements loaded']
default_material_value = ['C', 'Al'] if ('C' in ELEMENTS and 'Al' in ELEMENTS) else (list(ELEMENTS.keys())[:2] if ELEMENTS else [])

material_widget = widgets.SelectMultiple(
    options=default_material_options,
    value=default_material_value,
    description='Materials:',
    disabled=False,
)

pressure_widget = widgets.FloatSlider(
    value=64.0, min=0.0, max=120.0, step=0.5,
    description='Ice Pressure (GPa):',
    continuous_update=True,
)

p_scale_widget = widgets.FloatSlider(
    value=1.871, min=0.1, max=5.0, step=0.001,
    description='Ice P_scale:',
    continuous_update=True,
)

siphon_offset_widget = widgets.FloatSlider(
    value=0.0, min=-10.0, max=10.0, step=0.1,
    description='Siphon Angle Offset:',
    continuous_update=True,
)

std_err_widget = widgets.FloatSlider(
    value=0.05, min=0.001, max=0.1, step=0.001,
    description='Std Freq Error:',
    continuous_update=True,
)

opt_err_widget = widgets.FloatSlider(
    value=0.001, min=0.0001, max=0.01, step=0.0001,
    description='Opt Freq Error:',
    continuous_update=True,
)

dashboard = widgets.interactive(
    unified_t0c_dashboard,
    selected_materials=material_widget,
    pressure_gpa=pressure_widget,
    p_scale_ice=p_scale_widget,
    siphon_angle_offset=siphon_offset_widget,
    std_freq_err=std_err_widget,
    opt_freq_err=opt_err_widget,
)

display(dashboard)


In [ ]:
# @title Code Cell 3: Dedicated Sodium Siphon Deep Dive & Sensitivity Suite

# ---------- Shared helpers ----------

STANDARD_THETA = 109.0

def _compute_siphon_metrics(theta: float, siphon_theta: float, f_err: float) -> tuple[float, float, float]:
    """Core siphon metric using the current routing engine."""
    if t0c_engine is None:
        raise RuntimeError("T0CEngine not initialized")

    eta, _, _ = t0c_engine.select_eta(theta - siphon_theta, DEFAULT_D_CHI, f_err)
    cop = (eta ** NONLINEAR_COP_EXPONENT) * COP_BASE_MULTIPLIER
    heat_loss = (1.0 - eta) * DEFAULT_HEAT_LOSS_FACTOR
    return eta, cop, heat_loss


# ---------- Sodium Siphon Deep Dive ----------
def sodium_siphon_deep_dive(
    siphon_angle_offset: float = 0.0,
    std_f_err: float = 0.05,
    opt_f_err: float = 0.001
):
    """Interactive audit of Sodium Siphon performance."""
    siphon_theta = T0C_CONSTANTS["theta_siphon"] + siphon_angle_offset

    systems = {
        "Standard": {"theta": STANDARD_THETA, "f_err": std_f_err},
        "T0C-Optimized": {"theta": siphon_theta, "f_err": opt_f_err},
    }

    results = []
    for label, params in systems.items():
        eta, cop, heat_loss = _compute_siphon_metrics(
            theta=params["theta"],
            siphon_theta=siphon_theta,
            f_err=params["f_err"],
        )
        results.append({
            "System": label,
            "\u03B7": eta,
            "COP": cop,
            "Heat Loss (J)": heat_loss,
        })

    df = pd.DataFrame(results)

    # Plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    colors_cop = ["#666666", "#00ff88"]
    colors_heat = ["#ff4444", "#44aaff"]

    ax1.bar(df["System"], df["COP"], color=colors_cop, alpha=0.85)
    ax1.set_title("Coefficient of Performance")
    ax1.set_ylabel("COP (Relative Coherence Gain)")
    ax1.grid(axis="y", alpha=0.3)

    ax2.bar(df["System"], df["Heat Loss (J)"], color=colors_heat, alpha=0.85)
    ax2.set_title("Parasitic Heat Loss")
    ax2.set_ylabel("Joules")
    ax2.grid(axis="y", alpha=0.3)

    plt.suptitle("Sodium Siphon Performance Audit", fontsize=14, color="white")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    display(df.round(4))


# Interactive widget
deep_dive_widget = widgets.interactive(
    sodium_siphon_deep_dive,
    siphon_angle_offset=widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description="Siphon Angle Offset:"),
    std_f_err=widgets.FloatSlider(value=0.05, min=0.001, max=0.1, step=0.001, description="Std Freq Error:"),
    opt_f_err=widgets.FloatSlider(value=0.001, min=0.0001, max=0.01, step=0.0001, description="Opt Freq Error:"),
)
display(deep_dive_widget)


# ---------- η-Selector Gradient Analysis ----------
def demonstrate_gradients(
    dt_val: float = 0.1,
    dc_val: float = 0.01,
    df_val: float = 0.001,
):
    """Demonstrates autograd sensitivity."""
    if t0c_engine is None:
        print("Engine not initialized")
        return

    initial_eta, _, _ = t0c_engine.select_eta(dt_val, dc_val, df_val)

    # Gradients (autograd handles ArrayBox internally)
    grad_eta_wrt_theta = grad(eta_selector_autograd, 0)
    grad_eta_wrt_chi = grad(eta_selector_autograd, 1)
    grad_eta_wrt_f = grad(eta_selector_autograd, 2)

    grad_theta = float(grad_eta_wrt_theta(dt_val, dc_val, df_val))
    grad_chi = float(grad_eta_wrt_chi(dt_val, dc_val, df_val))
    grad_f = float(grad_eta_wrt_f(dt_val, dc_val, df_val))

    print("--- η Selector Gradients ---")
    print(f"Input values: Δθ={dt_val:.4f}°, Δχ={dc_val:.4f}, Δf={df_val:.4f}")
    print(f"Initial η: {initial_eta:.6f}")
    print(f"Gradient of η wrt Δθ: {grad_theta:.6f}")
    print(f"Gradient of η wrt Δχ: {grad_chi:.6f}")
    print(f"Gradient of η wrt Δf: {grad_f:.6f}")
    print("\nInterpretation:")
    print("  • Positive gradient → η increases as the parameter increases")
    print("  • Negative gradient → η decreases as the parameter increases")
    print("  • Magnitude shows sensitivity strength")

    # Perturbation plots
    perturbation_range = np.linspace(-0.01, 0.01, 50)
    # Vectorize perturbation calculations
    etas_theta = t0c_engine.select_eta(dt_val + perturbation_range, dc_val, df_val)[0]
    etas_chi = t0c_engine.select_eta(dt_val, dc_val + perturbation_range, df_val)[0]
    etas_f = t0c_engine.select_eta(dt_val, dc_val, df_val + perturbation_range)[0]

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"η Sensitivity to Small Perturbations (baseline η = {initial_eta:.4f})", fontsize=14, color="white")

    axs[0].plot(perturbation_range, etas_theta, color="cyan")
    axs[0].set_title("Δθ Sensitivity")
    axs[0].set_xlabel("Δθ perturbation")
    axs[0].set_ylabel("η")
    axs[0].grid(True, alpha=0.3)
    axs[0].axvline(0, color="red", ls="--", alpha=0.7)

    axs[1].plot(perturbation_range, etas_chi, color="magenta")
    axs[1].set_title("Δχ Sensitivity")
    axs[1].set_xlabel("Δχ perturbation")
    axs[1].set_ylabel("η")
    axs[1].grid(True, alpha=0.3)
    axs[1].axvline(0, color="red", ls="--", alpha=0.7)

    axs[2].plot(perturbation_range, etas_f, color="yellow")
    axs[2].set_title("Δf Sensitivity")
    axs[2].set_xlabel("Δf perturbation")
    axs[2].set_ylabel("η")
    axs[2].grid(True, alpha=0.3)
    axs[2].axvline(0, color="red", ls="--", alpha=0.7)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


# Gradient widgets
demo_dt = widgets.FloatSlider(value=0.1, min=-5.0, max=5.0, step=0.01, description="Δθ:")
demo_dc = widgets.FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001, description="Δχ:")
demo_df = widgets.FloatSlider(value=0.001, min=0.0001, max=0.01, step=0.0001, description="Δf:")

gradient_output = widgets.interactive_output(
    demonstrate_gradients, {"dt_val": demo_dt, "dc_val": demo_dc, "df_val": demo_df}
)
display(widgets.VBox([demo_dt, demo_dc, demo_df, gradient_output]))


# ---------- COP Sensitivity Analysis ----------
def calculate_cop_sensitivity(siphon_angle_offset_val: float = 0.0, opt_f_err_val: float = 0.001):
    """Numerical sensitivity of COP to standard frequency error."""
    standard_f_errors = np.linspace(0.001, 0.1, 50)
    cop_values = []

    siphon_theta = T0C_CONSTANTS["theta_siphon"] + siphon_angle_offset_val

    for sf_err in standard_f_errors:
        # eta, _, _ = t0c_engine.select_eta(STANDARD_THETA - siphon_theta, 0.01, sf_err)
        eta, _, _ = t0c_engine.select_eta(STANDARD_THETA - siphon_theta, DEFAULT_D_CHI, sf_err)
        cop = (eta ** NONLINEAR_COP_EXPONENT) * COP_BASE_MULTIPLIER
        cop_values.append(cop)

    cop_values = np.array(cop_values)
    sensitivity = np.diff(cop_values) / np.diff(standard_f_errors)

    fig, axs = plt.subplots(1, 2, figsize=(15, 6))

    axs[0].plot(standard_f_errors, cop_values, color="gold", marker="o", markersize=4)
    axs[0].set_title("COP vs Standard Frequency Error")
    axs[0].set_xlabel("Standard Freq Error")
    axs[0].set_ylabel("COP")
    axs[0].grid(True, alpha=0.3)

    axs[1].plot(standard_f_errors[:-1], sensitivity, color="red", marker="x", markersize=4, ls="--")
    axs[1].set_title("Sensitivity d(COP)/d(standard_f_err)")
    axs[1].set_xlabel("Standard Freq Error")
    axs[1].set_ylabel("Sensitivity")
    axs[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Average |sensitivity| of COP to std freq error: {np.mean(np.abs(sensitivity)):.4f}")


# Run once by default
calculate_cop_sensitivity()


In [ ]:
# @title
# ---------- T0C Engine Verification Test ----------
# Test T0C engine with known inputs to ensure basic functionality

if t0c_engine is None:
    print("❌ T0C Engine not initialized for verification.")
else:
    try:
        # Test at equilibrium (expect high eta)
        eta_test_eq, mode_test_eq, p_test_eq = t0c_engine.select_eta(0.0, 0.0, 0.0)
        assert 0.99 <= eta_test_eq <= 1.0, f"Invalid η at equilibrium: {eta_test_eq}"
        assert mode_test_eq == "PRECISION_RESOLVED" or mode_test_eq == "CCZ_AVERAGED", f"Unexpected mode at equilibrium: {mode_test_eq}"

        # Test with significant detuning (expect low eta)
        eta_test_detune, mode_test_detune, p_test_detune = t0c_engine.select_eta(5.0, 0.1, 0.05)
        assert 0.0 <= eta_test_detune < 0.5, f"Invalid η with detuning: {eta_test_detune}"

        print("✅ T0C Engine self-check passed: Basic η selection works as expected.")
    except Exception as e:
        print(f"❌ T0C Engine self-check failed: {e}")